In [ ]:

from pathlib import Path
from bs4 import BeautifulSoup
from bs4.filter import SoupStrainer

# **Scraping Process**

## Prerequisites
1. Open the IMDb review page: https://www.imdb.com/title/tt2527336/reviews/?ref_=tt_ururv_genai_sm
2. Click the "Show All" button to load all reviews.
3. Run the following script in the browser console to reveal spoiler content:

```javascript
    const spoilerButtons = document.querySelectorAll('.review-spoiler-button');
    spoilerButtons.forEach(button => {
        button.click();
    });
    console.log(`Clicked ${spoilerButtons.length} spoiler button(s).`);
```

4. Open inspector
5. Right click <html\> and copy InnerHTML
6. Paste to .html file and save to '/imdb_pages' folder

## Data Extraction Flow
1. Loads IMDb review pages in '/imdb_pages'
2. Parse the review HTML via BeautifulSoup4
3. Extracts review titles, content, and ratings
4. Store as stuctured dataset in /dataset

In [ ]:
dir_path = Path("imdb_pages")
file_paths = [str(file) for file in dir_path.iterdir() if file.is_file()]
file_paths

['imdb_pages/starwars-last-jedi.html',
 'imdb_pages/alien-vs-predator.html',
 'imdb_pages/backrooms.html',
 'imdb_pages/battlefield_earth.html',
 'imdb_pages/cats.html',
 'imdb_pages/conjuring.html',
 'imdb_pages/madame-web.html',
 'imdb_pages/matrix-resurrection.html',
 'imdb_pages/obsession.html',
 'imdb_pages/project-hail-mary.html']

In [ ]:
all_review_list = []

for path in file_paths:
    with open(path, "r", encoding="utf-8") as file:
        content = file.read()

        only_reviews = SoupStrainer("article")
        soup = BeautifulSoup(content, "lxml", parse_only=only_reviews)

        reviews = soup.find_all("article")
        review_list = []

        for rev in reviews:
            rating = rev.find("span", class_="ipc-rating-star--rating")
            title = rev.find("h4", class_="ipc-title__text")
            content = rev.find("div", class_="ipc-html-content-inner-div")
            
            review_list.append({
                'title': title.text if title else None,
                'content': content.text if content else None,
                'rating': int(rating.text) if rating else None,
            })
        
        all_review_list += review_list

df = pd.DataFrame(all_review_list)

In [ ]:
df.info()
df.to_csv('dataset/movie_reviews.csv', index=False)

<class 'pandas.DataFrame'>
RangeIndex: 25210 entries, 0 to 25209
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   title    25210 non-null  str    
 1   content  25120 non-null  str    
 2   rating   23959 non-null  float64
dtypes: float64(1), str(2)
memory usage: 591.0 KB


In [ ]:
df_grouped = df.groupby('rating', as_index=False).count()
df_sorted = df_grouped.sort_values(by='rating')

df_sorted.head(10)

,rating,title,content
0,1.0,4318,4284
1,2.0,1547,1536
2,3.0,1452,1444
3,4.0,1265,1264
4,5.0,1466,1465
5,6.0,1871,1871
6,7.0,2200,2200
7,8.0,2771,2748
8,9.0,2555,2543
9,10.0,4514,4514
